In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
from gensim.models import Word2Vec
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

In [ ]:
# === Download necessary NLTK data ===
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

FINE_TUNED_DIR = '../../../pretained_or_finetune-models'
REVIEWS_DATASET_DIR = '../../dataset'
UTILS_DIR = '../../utils'
NLTK_DATA_PATH = f"{FINE_TUNED_DIR}/nltk_data"

nltk.data.path.append(NLTK_DATA_PATH)

# === Constants ===
STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
CONST_VECTOR_SIZE = 50
CONST_DIM_WINDOWS = 3

In [ ]:
def advanced_clean_text(text):
    # Define domain-specific stopwords
    custom_stopwords = {'point', 'points', 'interest', 'landmark', 'landmarks', 'site', 'sites'}  # Remove generic tourism terms
    
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^a-z\s]', '', text)  # Remove special characters
    text = re.sub(r'\b(u|ur|b4)\b', 'you', text)  # Replace common abbreviations
    
    words = word_tokenize(text)
    words = [LEMMATIZER.lemmatize(word) for word in words if word not in STOPWORDS]
    words = [word for word in words if word not in custom_stopwords and word not in string.punctuation]  # Remove tourism-related stopwords

    return ' '.join(words)

In [ ]:
combined_details_df = pd.read_csv("../data/combined_details.csv")
combined_details_df['tags'] = combined_details_df['tags'].fillna('')
combined_details_df['tags'].replace("", "other", inplace=True)

In [ ]:
combined_details_df['cleaned_tags'] = combined_details_df['tags'].apply(advanced_clean_text)

In [ ]:
sentences = combined_details_df['cleaned_tags'].str.split()  # Tokenized tags
word2vec_model = Word2Vec(sentences, vector_size=CONST_VECTOR_SIZE, window=CONST_DIM_WINDOWS, min_count=1, workers=4)

# Compute TF-IDF
tag_texts = [' '.join(tag) for tag in sentences]
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(tag_texts)
tfidf_vocab = tfidf_vectorizer.vocabulary_

In [ ]:
def get_tag_vector(tag):
    vectors = []
    for word in tag:
        if word in word2vec_model.wv and word in tfidf_vocab:
            tfidf_weight = tfidf_matrix[0, tfidf_vocab[word]]  # Get TF-IDF weight for the word
            word_vector = word2vec_model.wv[word] * tfidf_weight  # Multiply Word2Vec by TF-IDF weight
            vectors.append(word_vector)
    if vectors:
        return np.mean(vectors, axis=0)  # Average the weighted Word2Vec embeddings
    else:
        return np.zeros(word2vec_model.vector_size)  # Return a zero vector if no valid words

# Generate tag embeddings
tag_embeddings = np.array([get_tag_vector(tag) for tag in sentences])